## Imports:

In [1]:
import pandas as pd
import optuna
import torch
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
import numpy as np
from lightgbm import early_stopping
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from scipy.optimize import minimize
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import OrdinalEncoder


## Data loading:

In [2]:
train_data = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')

## Data preprocessing:

In [3]:
train_data.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [4]:
train_data.isna().sum()

id                             0
health_condition               0
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64

In [6]:
# NUMERICAL DATA

num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
train_medians = train_data[num_cols].median()

for col in num_cols:
    train_data[f'{col}_was_missing'] = train_data[col].isnull().astype(int)
    test_data[f'{col}_was_missing'] = test_data[col].isnull().astype(int)

for df in [train_data, test_data]:
    if 'calorie_expenditure' in df.columns:
        df['calorie_cat'] = (df['calorie_expenditure'] // 5).fillna(-1).astype(str)
    if 'water_intake' in df.columns:
        df['water_cat'] = (df['water_intake'] * 50).fillna(-1).astype(int).astype(str)
    if 'step_count' in df.columns:
        df['step_count_rounded'] = df['step_count'].round(-1).fillna(-1).astype(str)

bin_config = {'sleep_duration':[70], 'water_intake':[10]}
category_map_bins = {}

for col, bins_list in bin_config.items():
    if col in train_data.columns:
        for n_bins in bins_list:
            bin_name = f"{col}_{n_bins}_quantile_bin_"
            kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile', subsample=None)
            
            train_data[bin_name] = kb.fit_transform(train_data[[col]].fillna(train_medians[col])).ravel().astype('int32').astype(str)
            category_map_bins[bin_name] = kb
            
            if col in test_data.columns:
                test_data[bin_name] = category_map_bins[bin_name].transform(test_data[[col]].fillna(train_medians[col])).ravel().astype('int32').astype(str)

train_data[num_cols] = train_data[num_cols].fillna(train_medians)
test_data[num_cols] = test_data[num_cols].fillna(train_medians)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(


In [7]:
# CATEGORICAL DATA

cat_cols = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']

new_features = ['calorie_cat', 'water_cat', 'step_count_rounded', 'sleep_duration_70_quantile_bin_', 'water_intake_10_quantile_bin_']
cat_cols.extend(new_features)

train_data[cat_cols] = train_data[cat_cols].fillna('Missing')
test_data[cat_cols] = test_data[cat_cols].fillna('Missing')

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

train_data[cat_cols] = encoder.fit_transform(train_data[cat_cols].astype(str))
test_data[cat_cols] = encoder.transform(test_data[cat_cols].astype(str))

print(f"[INFO] Total categorical features encoded: {len(cat_cols)}")

[INFO] Total categorical features encoded: 11


In [9]:
train_data.isna().sum()

id                                 0
health_condition                   0
sleep_duration                     0
heart_rate                         0
bmi                                0
calorie_expenditure                0
step_count                         0
exercise_duration                  0
water_intake                       0
diet_type                          0
stress_level                       0
sleep_quality                      0
physical_activity_level            0
smoking_alcohol                    0
gender                             0
sleep_duration_was_missing         0
heart_rate_was_missing             0
bmi_was_missing                    0
calorie_expenditure_was_missing    0
step_count_was_missing             0
exercise_duration_was_missing      0
water_intake_was_missing           0
calorie_cat                        0
water_cat                          0
step_count_rounded                 0
sleep_duration_70_quantile_bin_    0
water_intake_10_quantile_bin_      0
d

In [10]:
train_data.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,...,bmi_was_missing,calorie_expenditure_was_missing,step_count_was_missing,exercise_duration_was_missing,water_intake_was_missing,calorie_cat,water_cat,step_count_rounded,sleep_duration_70_quantile_bin_,water_intake_10_quantile_bin_
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,3.0,...,0,0,0,0,0,195.0,200.0,364.0,34.0,2.0
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,2.0,...,0,0,0,0,0,154.0,170.0,1385.0,56.0,0.0
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,3.0,...,0,0,0,0,0,298.0,187.0,466.0,34.0,1.0
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,3.0,...,0,0,0,0,0,287.0,2.0,1113.0,1.0,3.0
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,3.0,...,0,0,0,0,0,273.0,13.0,1054.0,32.0,5.0


In [11]:
# FEATURES AND LABEL CREATING

# ---------------------------------------------------------
# y_train creating
# ---------------------------------------------------------
y_train_raw = train_data['health_condition']
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)

# ---------------------------------------------------------
# X_train and X_test creating
# ---------------------------------------------------------
X_train = train_data.drop(columns=['id', 'health_condition'])
X_test = test_data.drop(columns=['id'])

## Models creating:

In [12]:
# ENSEMBLE

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgb_test_total = np.zeros((len(X_test), 3))
cat_test_total = np.zeros((len(X_test), 3))
xgb_test_total = np.zeros((len(X_test), 3))

print("Training is starting...")

for fold, (train_idxs, val_idxs) in enumerate(kf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idxs], y_train[train_idxs]
    X_va, y_va = X_train.iloc[val_idxs], y_train[val_idxs]

# ---------------------------------------------------------
# LightGBM
# ---------------------------------------------------------
    lgb_model = LGBMClassifier(
        n_estimators=1500, 
        learning_rate=0.05, 
        max_depth=6, 
        num_leaves=31, 
        class_weight='balanced', 
        subsample=0.8, 
        colsample_bytree=0.8, 
        random_state=42, 
        n_jobs=-1, 
        verbose=-1)
    
    lgb_model.fit(
        X_tr, 
        y_tr, 
        eval_set=[(X_va, y_va)], 
        callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    lgb_test_total += lgb_model.predict_proba(X_test) / kf.n_splits

# ---------------------------------------------------------
# CatBoost 
# ---------------------------------------------------------
    cat_model = CatBoostClassifier(
        iterations=1500, 
        learning_rate=0.05, 
        depth=6, 
        l2_leaf_reg=5, 
        auto_class_weights='Balanced', 
        random_seed=42, 
        early_stopping_rounds=50, 
        verbose=0
    )
    
    cat_model.fit(
        X_tr, 
        y_tr, 
        eval_set=(X_va, y_va)
    )
    
    cat_test_total += cat_model.predict_proba(X_test) / kf.n_splits

# ---------------------------------------------------------
# XGBoost
# ---------------------------------------------------------

    sample_weight = compute_sample_weight(class_weight='balanced', y=y_tr)
    
    xgb_model = XGBClassifier(
        n_estimators=1500, 
        learning_rate=0.05, 
        max_depth=6, 
        subsample=0.8, 
        colsample_bytree=0.8, 
        random_state=42, 
        n_jobs=-1, 
        eval_metric='mlogloss', 
        early_stopping_rounds=50
    ) 
    
    xgb_model.fit(
        X_tr, 
        y_tr, 
        sample_weight=sample_weight, 
        eval_set=[(X_va, y_va)], 
        verbose=False
    )
    
    xgb_test_total += xgb_model.predict_proba(X_test) / kf.n_splits

    print(f"Fold {fold + 1}/5 succesfully completed!")

Training is starting...
Fold 1/5 succesfully completed!
Fold 2/5 succesfully completed!
Fold 3/5 succesfully completed!
Fold 4/5 succesfully completed!
Fold 5/5 succesfully completed!


In [14]:
# BLENDING

final_probs = ( 0.45 * cat_test_total + 0.45 * lgb_test_total + 0.10 * xgb_test_total ) 
final_classes = np.argmax(final_probs, axis=1) 
final_labels = label_encoder.inverse_transform(final_classes) 

In [15]:
# CREATING SIBMISSION FILE

submission = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv') 
submission['health_condition'] = final_labels 
submission.to_csv('submission.csv', index=False) 
print("\nfile'submission.csv' is ready")


file'submission.csv' is ready
